# RQ5: Winter-Wheat Yield Prediction Using Crop-Pure and Mixed-Pixel SIF

**Research question:** To what extent does high-resolution, crop-pure enhanced SIF improve winter-wheat yield prediction RMSE compared with mixed-pixel SIF derived from raw satellite observations?

The notebook first compares nine fixed Random Forest yield models on exactly the same NUTS3 region-year observations. Every model uses the same winter-wheat fraction, temporal split and Random Forest settings. No hyperparameter tuning is performed.

- Crop-pure model: separate calibrated SIF predictors for March-July.
- Four mixed-pixel monthly models: all accepted footprints and footprints with at least 5%, 10% or 30% winter-wheat coverage.
- Four mixed-pixel seasonal-mean models: one observed-month mean SIF predictor for each of the same coverage definitions.

Models are trained on 2019-2022 and evaluated once on the untouched 2024 observations.

## 1. Imports and Configuration

In [ ]:
from pathlib import Path
import json
import random

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

DATA_PATH = Path(
    'data/winter_wheat_yield_model/ww_model_data.csv'
)
OUTPUT_DIR = Path(
    'data/winter_wheat_yield_model/rq5_results'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_YEARS = [2019, 2020, 2021, 2022]
TEST_YEAR = 2024
MONTHS = ['March', 'April', 'May', 'June', 'July']
BOOTSTRAP_REPETITIONS = 10_000

RF_N_ESTIMATORS = 500
RF_MAX_DEPTH = 5
RF_MIN_SAMPLES_LEAF = 3

print('Input:', DATA_PATH)
print('Outputs:', OUTPUT_DIR.resolve())

## 2. Load and Validate the Final Modeling Table

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)

data = pd.read_csv(DATA_PATH)

identifier_columns = ['nuts_id', 'nuts3', 'mgrs_tile', 'year']
crop_pure_sif_columns = [
    f'SIF_{month}_calibrated' for month in MONTHS
]
raw_monthly_sif_columns = {
    'raw_all': [f'raw_all_SIF_{month}' for month in MONTHS],
    'raw_ww05': [f'raw_ww05_SIF_{month}' for month in MONTHS],
    'raw_ww10': [f'raw_ww10_SIF_{month}' for month in MONTHS],
    'raw_ww30': [f'raw_ww30_SIF_{month}' for month in MONTHS],
}
raw_seasonal_sif_columns = {
    'raw_all': 'raw_all_seasonal_sif',
    'raw_ww05': 'raw_ww05_seasonal_sif',
    'raw_ww10': 'raw_ww10_seasonal_sif',
    'raw_ww30': 'raw_ww30_seasonal_sif',
}
all_raw_monthly_columns = [
    column
    for columns in raw_monthly_sif_columns.values()
    for column in columns
]
required_columns = (
    identifier_columns
    + crop_pure_sif_columns
    + all_raw_monthly_columns
    + list(raw_seasonal_sif_columns.values())
    + ['ww_pct', 'ww_yield']
)

missing_columns = sorted(set(required_columns) - set(data.columns))
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

numeric_columns = (
    ['year']
    + crop_pure_sif_columns
    + all_raw_monthly_columns
    + list(raw_seasonal_sif_columns.values())
    + ['ww_pct', 'ww_yield']
)
for column in numeric_columns:
    data[column] = pd.to_numeric(data[column], errors='coerce')

data = data.loc[data['year'].isin(TRAIN_YEARS + [TEST_YEAR])].copy()
data['year'] = data['year'].astype(int)

duplicate_keys = data.duplicated(['nuts_id', 'year'], keep=False)
if duplicate_keys.any():
    display(data.loc[duplicate_keys, identifier_columns])
    raise ValueError('Expected one row per NUTS3 region and year.')

finite_required = (
    crop_pure_sif_columns
    + all_raw_monthly_columns
    + list(raw_seasonal_sif_columns.values())
    + ['ww_pct', 'ww_yield']
)
nonfinite_counts = {
    column: int((~np.isfinite(data[column])).sum())
    for column in finite_required
}
nonfinite_counts = {
    column: count for column, count in nonfinite_counts.items()
    if count > 0
}
if nonfinite_counts:
    raise ValueError(
        'The final modeling table still contains non-finite values: '
        f'{nonfinite_counts}'
    )

print('Rows:', len(data))
display(data.groupby('year').size().rename('n_rows').to_frame())
display(data[required_columns].head())

## 3. Define Nine Predictor Sets and the Fixed Temporal Split

All nine models use only their corresponding SIF predictors. `ww_pct` is retained in exported metadata tables for interpretation, but it is not supplied to any Random Forest. The models differ only in their SIF representation: crop-pure monthly SIF, four raw monthly coverage variants, or four raw seasonal-mean coverage variants.

In [ ]:
shared_feature_columns = []

model_specs = {
    'crop_pure_monthly': {
        'label': 'Crop-pure monthly SIF',
        'sif_representation': 'Five calibrated crop-pure monthly SIF predictors',
        'sif_features': crop_pure_sif_columns,
    },
    'raw_all_monthly': {
        'label': 'Raw all-footprint monthly SIF',
        'sif_representation': 'Five monthly raw SIF predictors; all accepted footprints',
        'sif_features': raw_monthly_sif_columns['raw_all'],
    },
    'raw_ww05_monthly': {
        'label': 'Raw >=5% wheat monthly SIF',
        'sif_representation': 'Five monthly raw SIF predictors; footprint wheat >=5%',
        'sif_features': raw_monthly_sif_columns['raw_ww05'],
    },
    'raw_ww10_monthly': {
        'label': 'Raw >=10% wheat monthly SIF',
        'sif_representation': 'Five monthly raw SIF predictors; footprint wheat >=10%',
        'sif_features': raw_monthly_sif_columns['raw_ww10'],
    },
    'raw_ww30_monthly': {
        'label': 'Raw >=30% wheat monthly SIF',
        'sif_representation': 'Five monthly raw SIF predictors; footprint wheat >=30%',
        'sif_features': raw_monthly_sif_columns['raw_ww30'],
    },
    'raw_all_seasonal_mean': {
        'label': 'Raw all-footprint seasonal-mean SIF',
        'sif_representation': 'One March-July observed-month mean; all accepted footprints',
        'sif_features': [raw_seasonal_sif_columns['raw_all']],
    },
    'raw_ww05_seasonal_mean': {
        'label': 'Raw >=5% wheat seasonal-mean SIF',
        'sif_representation': 'One March-July observed-month mean; footprint wheat >=5%',
        'sif_features': [raw_seasonal_sif_columns['raw_ww05']],
    },
    'raw_ww10_seasonal_mean': {
        'label': 'Raw >=10% wheat seasonal-mean SIF',
        'sif_representation': 'One March-July observed-month mean; footprint wheat >=10%',
        'sif_features': [raw_seasonal_sif_columns['raw_ww10']],
    },
    'raw_ww30_seasonal_mean': {
        'label': 'Raw >=30% wheat seasonal-mean SIF',
        'sif_representation': 'One March-July observed-month mean; footprint wheat >=30%',
        'sif_features': [raw_seasonal_sif_columns['raw_ww30']],
    },
}

for spec in model_specs.values():
    spec['features'] = spec['sif_features'] + shared_feature_columns

train_mask = data['year'].isin(TRAIN_YEARS)
test_mask = data['year'].eq(TEST_YEAR)

train_data = data.loc[train_mask].copy().reset_index(drop=True)
test_data = data.loc[test_mask].copy().reset_index(drop=True)

X_train_by_model = {
    key: train_data[spec['features']].copy()
    for key, spec in model_specs.items()
}
X_test_by_model = {
    key: test_data[spec['features']].copy()
    for key, spec in model_specs.items()
}

y_train = train_data['ww_yield'].to_numpy(dtype=np.float64)
y_test = test_data['ww_yield'].to_numpy(dtype=np.float64)
for key in model_specs:
    assert np.isfinite(X_train_by_model[key].to_numpy()).all(), key
    assert np.isfinite(X_test_by_model[key].to_numpy()).all(), key

print('Training rows:', len(train_data))
print('Test rows:', len(test_data))
model_design = pd.DataFrame([
    {
        'model_key': key,
        'model': spec['label'],
        'n_features': len(spec['features']),
        'sif_representation': spec['sif_representation'],
        'features': ';'.join(spec['features']),
    }
    for key, spec in model_specs.items()
])
display(model_design)

## 4. Fixed Random Forest Configuration

All nine models use the same fixed, small-sample-regularized Random Forest settings. Tree depth and minimum leaf size are constrained to reduce overfitting; the intended differences remain their SIF coverage threshold and monthly-versus-seasonal representation.

In [ ]:
def rmse(observed, predicted):
    return float(np.sqrt(mean_squared_error(observed, predicted)))

def make_random_forest():
    return RandomForestRegressor(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        random_state=SEED,
        n_jobs=-1,
    )

fixed_model_settings = {
    'n_estimators': RF_N_ESTIMATORS,
    'max_depth': RF_MAX_DEPTH,
    'min_samples_leaf': RF_MIN_SAMPLES_LEAF,
    'random_state': SEED,
    'n_jobs': -1,
}
display(pd.DataFrame([fixed_model_settings]))

## 5. Fit All Nine Models and Predict the Held-Out 2024 Regions

In [ ]:
def regression_metrics(observed, predicted):
    observed = np.asarray(observed, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)
    residual = predicted - observed
    observed_sd = float(np.std(observed, ddof=1))

    if len(observed) >= 2 and np.var(observed) > 0:
        slope, intercept = np.polyfit(observed, predicted, 1)
    else:
        slope, intercept = np.nan, np.nan

    model_rmse = rmse(observed, predicted)
    return {
        'n': int(len(observed)),
        'rmse': model_rmse,
        'normalized_rmse': (
            model_rmse / observed_sd if observed_sd > 0 else np.nan
        ),
        'mae': float(mean_absolute_error(observed, predicted)),
        'r2': float(r2_score(observed, predicted)),
        'bias': float(np.mean(residual)),
        'regression_slope': float(slope),
        'regression_intercept': float(intercept),
        'residual_sd': float(np.std(residual, ddof=1)),
    }

fitted_models = {}
test_prediction_by_model = {}
metric_rows = []

for key, spec in model_specs.items():
    print(f"Fitting {key}: {spec['label']}")
    model = make_random_forest()
    model.fit(X_train_by_model[key], y_train)
    prediction = model.predict(X_test_by_model[key])

    fitted_models[key] = model
    test_prediction_by_model[key] = prediction
    metric_rows.append({
        'model_key': key,
        'model': spec['label'],
        'sif_representation': spec['sif_representation'],
        'n_train': len(train_data),
        'n_test': len(test_data),
        'n_features': len(spec['features']),
        **regression_metrics(y_test, prediction),
    })

nine_model_metrics = (
    pd.DataFrame(metric_rows)
    .sort_values(['rmse', 'mae'])
    .reset_index(drop=True)
)
nine_model_metrics.insert(0, 'rmse_rank', np.arange(1, len(nine_model_metrics) + 1))
nine_model_metrics.to_csv(
    OUTPUT_DIR / 'nine_model_test_metrics.csv',
    index=False,
)

nine_model_test_predictions = test_data[
    identifier_columns + ['ww_pct']
].copy()
nine_model_test_predictions['observed_ww_yield'] = y_test
for key, prediction in test_prediction_by_model.items():
    nine_model_test_predictions[f'predicted_{key}'] = prediction
    nine_model_test_predictions[f'residual_{key}'] = prediction - y_test
    nine_model_test_predictions[f'absolute_error_{key}'] = np.abs(
        prediction - y_test
    )
nine_model_test_predictions.to_csv(
    OUTPUT_DIR / 'nine_model_test_predictions.csv',
    index=False,
)

display(nine_model_metrics)
display(nine_model_test_predictions.head())
print(
    'Pause after the feature-importance section to select the primary '
    'mixed-pixel comparator before updating the paired bootstrap and RQ5 outputs.'
)

## 6. Predictor Importance for All Nine Random Forests

The tables rank predictors using each fitted forest's impurity-based feature importance. Importances sum to one within each model; compare ranks within a model rather than raw importance magnitudes across models.

In [ ]:
def feature_importance_table(model, feature_columns, model_name):
    table = pd.DataFrame({
        'model': model_name,
        'predictor': feature_columns,
        'importance': model.feature_importances_,
    }).sort_values('importance', ascending=False).reset_index(drop=True)
    table.insert(1, 'rank', np.arange(1, len(table) + 1))
    return table

feature_importance_tables = {}
for key, spec in model_specs.items():
    table = feature_importance_table(
        fitted_models[key],
        spec['features'],
        spec['label'],
    )
    table.insert(0, 'model_key', key)
    feature_importance_tables[key] = table

feature_importance = pd.concat(
    feature_importance_tables.values(),
    ignore_index=True,
)
feature_importance.to_csv(
    OUTPUT_DIR / 'random_forest_feature_importance.csv',
    index=False,
)

for key, table in feature_importance_tables.items():
    print(f"{model_specs[key]['label']} predictor importance:")
    display(table)

## 7. Paired Region Bootstrap

The bootstrap resamples the same 2024 NUTS3 rows for all three selected models. Two paired comparisons are calculated: raw all-footprint minus crop-pure RMSE, and raw >=5% wheat minus crop-pure RMSE. Positive improvement means the crop-pure model has lower RMSE.

In [ ]:
selected_model_keys = [
    'crop_pure_monthly',
    'raw_ww05_monthly',
    'raw_all_monthly',
]
comparison_specs = {
    'raw_ww05_minus_crop_pure': {
        'raw_key': 'raw_ww05_monthly',
        'label': 'Raw >=5% wheat minus crop-pure',
    },
    'raw_all_minus_crop_pure': {
        'raw_key': 'raw_all_monthly',
        'label': 'Raw all-footprint minus crop-pure',
    },
}
crop_pure_key = 'crop_pure_monthly'
selected_metrics = {
    key: regression_metrics(y_test, test_prediction_by_model[key])
    for key in selected_model_keys
}

rng = np.random.default_rng(SEED)
bootstrap_rows = []
n_test = len(y_test)

for repetition in range(BOOTSTRAP_REPETITIONS):
    sample_index = rng.integers(0, n_test, size=n_test)
    observed_sample = y_test[sample_index]
    row = {'repetition': repetition + 1}

    for key in selected_model_keys:
        prediction_sample = test_prediction_by_model[key][sample_index]
        row[f'{key}_rmse'] = rmse(observed_sample, prediction_sample)

    crop_rmse = row[f'{crop_pure_key}_rmse']
    for comparison_key, comparison in comparison_specs.items():
        raw_rmse = row[f"{comparison['raw_key']}_rmse"]
        improvement = raw_rmse - crop_rmse
        row[f'{comparison_key}_rmse_improvement'] = improvement
        row[f'{comparison_key}_rmse_reduction_pct'] = (
            100 * improvement / raw_rmse if raw_rmse > 0 else np.nan
        )
    bootstrap_rows.append(row)

bootstrap_results = pd.DataFrame(bootstrap_rows)
bootstrap_results.to_csv(
    OUTPUT_DIR / 'bootstrap_rmse_improvement_draws.csv', index=False
)

def percentile_interval(values):
    values = np.asarray(values, dtype=np.float64)
    return tuple(np.quantile(values, [0.025, 0.975]))

model_rmse_ci = {
    key: percentile_interval(bootstrap_results[f'{key}_rmse'])
    for key in selected_model_keys
}
bootstrap_summary_rows = []
for comparison_key, comparison in comparison_specs.items():
    raw_key = comparison['raw_key']
    improvement_column = f'{comparison_key}_rmse_improvement'
    reduction_column = f'{comparison_key}_rmse_reduction_pct'
    improvement_ci = percentile_interval(bootstrap_results[improvement_column])
    reduction_ci = percentile_interval(bootstrap_results[reduction_column])
    observed_improvement = (
        selected_metrics[raw_key]['rmse']
        - selected_metrics[crop_pure_key]['rmse']
    )
    bootstrap_summary_rows.append({
        'comparison_key': comparison_key,
        'comparison': comparison['label'],
        'bootstrap_repetitions': BOOTSTRAP_REPETITIONS,
        'observed_rmse_improvement_t_ha': observed_improvement,
        'mean_bootstrap_rmse_improvement_t_ha': float(
            bootstrap_results[improvement_column].mean()
        ),
        'rmse_improvement_ci_lower': improvement_ci[0],
        'rmse_improvement_ci_upper': improvement_ci[1],
        'mean_rmse_reduction_pct': float(
            bootstrap_results[reduction_column].mean()
        ),
        'rmse_reduction_pct_ci_lower': reduction_ci[0],
        'rmse_reduction_pct_ci_upper': reduction_ci[1],
        'probability_crop_pure_lower_rmse': float(
            np.mean(bootstrap_results[improvement_column] > 0)
        ),
    })
bootstrap_summary = pd.DataFrame(bootstrap_summary_rows)
bootstrap_summary.to_csv(
    OUTPUT_DIR / 'bootstrap_rmse_improvement_summary.csv', index=False
)
display(bootstrap_summary)

## Output 1. Model Performance Comparison Table

In [ ]:
performance_rows = []
for key in selected_model_keys:
    metrics = selected_metrics[key]
    ci = model_rmse_ci[key]
    performance_rows.append({
        'model_key': key,
        'model': model_specs[key]['label'],
        'sif_predictors': ';'.join(model_specs[key]['sif_features']),
        'n_train': len(train_data),
        'n_test': len(test_data),
        **metrics,
        'rmse_bootstrap_ci_lower': ci[0],
        'rmse_bootstrap_ci_upper': ci[1],
    })
performance_table = pd.DataFrame(performance_rows)

comparison_rows = []
for comparison_key, comparison in comparison_specs.items():
    raw_key = comparison['raw_key']
    raw_metrics = selected_metrics[raw_key]
    crop_metrics = selected_metrics[crop_pure_key]
    bootstrap_row = bootstrap_summary.loc[
        bootstrap_summary['comparison_key'].eq(comparison_key)
    ].iloc[0]
    rmse_improvement = raw_metrics['rmse'] - crop_metrics['rmse']
    comparison_rows.append({
        'comparison_key': comparison_key,
        'comparison': comparison['label'],
        'rmse_improvement_t_ha': rmse_improvement,
        'rmse_reduction_pct': (
            100 * rmse_improvement / raw_metrics['rmse']
            if raw_metrics['rmse'] > 0 else np.nan
        ),
        'mae_improvement_t_ha': raw_metrics['mae'] - crop_metrics['mae'],
        'r2_improvement': crop_metrics['r2'] - raw_metrics['r2'],
        'rmse_improvement_ci_lower': bootstrap_row['rmse_improvement_ci_lower'],
        'rmse_improvement_ci_upper': bootstrap_row['rmse_improvement_ci_upper'],
        'probability_crop_pure_lower_rmse': (
            bootstrap_row['probability_crop_pure_lower_rmse']
        ),
    })
comparison_table = pd.DataFrame(comparison_rows)

performance_table.to_csv(
    OUTPUT_DIR / 'yield_model_performance_comparison.csv',
    index=False,
)
comparison_table.to_csv(
    OUTPUT_DIR / 'yield_model_rmse_improvement.csv',
    index=False,
)

display(performance_table)
display(comparison_table)

## Output 2. Observed Versus Predicted Yield

In [ ]:
plot_values = np.concatenate(
    [y_test]
    + [test_prediction_by_model[key] for key in selected_model_keys]
)
plot_min = float(np.min(plot_values))
plot_max = float(np.max(plot_values))
plot_padding = max((plot_max - plot_min) * 0.08, 0.5)
axis_limits = (plot_min - plot_padding, plot_max + plot_padding)

figure, axes = plt.subplots(1, 3, figsize=(18.0, 5.8), sharex=True, sharey=True)

plot_specs = [
    (
        axes[0],
        test_prediction_by_model['crop_pure_monthly'],
        selected_metrics['crop_pure_monthly'],
        'Crop-pure monthly SIF',
        '#4C78A8',
    ),
    (
        axes[1],
        test_prediction_by_model['raw_ww05_monthly'],
        selected_metrics['raw_ww05_monthly'],
        'Raw >=5% wheat monthly SIF',
        '#4C78A8',
    ),
    (
        axes[2],
        test_prediction_by_model['raw_all_monthly'],
        selected_metrics['raw_all_monthly'],
        'Raw all-footprint monthly SIF',
        '#4C78A8',
    ),
]

for axis, prediction, metrics, title, color in plot_specs:
    axis.scatter(
        y_test,
        prediction,
        s=48,
        color=color,
        edgecolor='white',
        linewidth=0.6,
        alpha=0.9,
    )
    axis.plot(axis_limits, axis_limits, '--', color='black', linewidth=1.5, label='y = x')
    regression_x = np.asarray(axis_limits)
    regression_y = (
        metrics['regression_slope'] * regression_x
        + metrics['regression_intercept']
    )
    axis.plot(regression_x, regression_y, color='#D62728', linewidth=2, label='regression')
    axis.set_xlim(axis_limits)
    axis.set_ylim(axis_limits)
    axis.set_aspect('equal', adjustable='box')
    axis.grid(alpha=0.25)
    axis.set_title(title, fontsize=13, fontweight='bold')
    axis.set_xlabel('Observed winter-wheat yield (t/ha)')
    axis.text(
        0.04,
        0.96,
        (
            f"RMSE={metrics['rmse']:.2f} t/ha\n"
            f"R2={metrics['r2']:.3f}\n"
            f"slope={metrics['regression_slope']:.3f}\n"
            f"N={metrics['n']}"
        ),
        transform=axis.transAxes,
        va='top',
        bbox={'facecolor': 'white', 'alpha': 0.85, 'edgecolor': 'none'},
    )
    axis.legend(loc='lower right', frameon=False)

axes[0].set_ylabel('Predicted winter-wheat yield (t/ha)')
figure.suptitle('Held-out 2024 NUTS3 yield predictions', fontsize=15, fontweight='bold')
figure.tight_layout()
figure.savefig(
    OUTPUT_DIR / 'observed_vs_predicted_yield_models.pdf',
    bbox_inches='tight',
)
plt.show()

## Output 3. Paired Regional Absolute-Error Improvement

Positive values indicate lower absolute error from the crop-pure SIF model. Both raw baselines are compared with the same crop-pure prediction for every held-out region.

In [ ]:
regional_errors = test_data[identifier_columns + ['ww_pct']].copy()
regional_errors['observed_ww_yield'] = y_test
regional_errors['absolute_error_crop_pure'] = np.abs(
    test_prediction_by_model['crop_pure_monthly'] - y_test
)
regional_errors['absolute_error_raw_ww05'] = np.abs(
    test_prediction_by_model['raw_ww05_monthly'] - y_test
)
regional_errors['absolute_error_raw_all'] = np.abs(
    test_prediction_by_model['raw_all_monthly'] - y_test
)
regional_errors['improvement_raw_ww05_minus_crop'] = (
    regional_errors['absolute_error_raw_ww05']
    - regional_errors['absolute_error_crop_pure']
)
regional_errors['improvement_raw_all_minus_crop'] = (
    regional_errors['absolute_error_raw_all']
    - regional_errors['absolute_error_crop_pure']
)
regional_errors['mean_crop_pure_improvement'] = regional_errors[[
    'improvement_raw_ww05_minus_crop',
    'improvement_raw_all_minus_crop',
]].mean(axis=1)
regional_errors = regional_errors.sort_values(
    'mean_crop_pure_improvement', ascending=True
).reset_index(drop=True)
regional_errors.to_csv(
    OUTPUT_DIR / 'paired_nuts3_absolute_errors_2024.csv', index=False
)

panel_specs = [
    ('improvement_raw_ww05_minus_crop', 'Raw >=5% wheat error - crop-pure error'),
    ('improvement_raw_all_minus_crop', 'Raw all-footprint error - crop-pure error'),
]
figure_height = min(max(8.0, 0.28 * len(regional_errors)), 18.0)
figure, axes = plt.subplots(
    1, 2, figsize=(16.0, figure_height), sharey=True
)
for axis, (column, title) in zip(axes, panel_specs):
    bar_colors = np.where(
        regional_errors[column] >= 0, '#2A9D8F', '#D95F59'
    )
    axis.barh(
        regional_errors['nuts3'], regional_errors[column],
        color=bar_colors, edgecolor='white', linewidth=0.5,
    )
    axis.axvline(0, color='black', linewidth=1.2)
    axis.set_xlabel('Absolute-error improvement (t/ha)')
    axis.set_title(title, fontsize=12, fontweight='bold')
    axis.grid(axis='x', alpha=0.25)
    axis.text(
        0.98, 0.02,
        f"Improved: {int((regional_errors[column] > 0).sum())}/{len(regional_errors)}",
        transform=axis.transAxes, ha='right', va='bottom',
        bbox={'facecolor': 'white', 'alpha': 0.85, 'edgecolor': 'none'},
    )
axes[0].set_ylabel('NUTS3 region')
figure.suptitle(
    'Crop-pure SIF yield-model improvement by region (2024)',
    fontsize=14, fontweight='bold',
)
figure.tight_layout()
figure.savefig(
    OUTPUT_DIR / 'paired_regional_absolute_error_improvement.pdf',
    bbox_inches='tight',
)
plt.show()
display(regional_errors)

## Output 4. Bootstrap Distributions of RMSE Improvement

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14.5, 5.7), sharey=True)
for axis, (comparison_key, comparison) in zip(
    axes, comparison_specs.items()
):
    improvement_column = f'{comparison_key}_rmse_improvement'
    improvement_values = bootstrap_results[improvement_column].to_numpy()
    improvement_mean = float(np.mean(improvement_values))
    improvement_ci = percentile_interval(improvement_values)
    probability_improved = float(np.mean(improvement_values > 0))

    axis.hist(
        improvement_values, bins=55, color='#4C78A8',
        edgecolor='white', linewidth=0.45,
    )
    axis.axvline(
        0, color='black', linestyle='--', linewidth=1.6,
        label='No improvement',
    )
    axis.axvline(
        improvement_mean, color='#D62728', linewidth=2,
        label=f'Mean = {improvement_mean:.3f} t/ha',
    )
    axis.axvline(
        improvement_ci[0], color='#D62728', linestyle=':', linewidth=1.5,
    )
    axis.axvline(
        improvement_ci[1], color='#D62728', linestyle=':', linewidth=1.5,
        label=f'95% CI [{improvement_ci[0]:.3f}, {improvement_ci[1]:.3f}]',
    )
    axis.set_xlabel('Raw RMSE - crop-pure RMSE (t/ha)')
    axis.set_title(comparison['label'], fontsize=12, fontweight='bold')
    axis.text(
        0.98, 0.95,
        f'P(crop-pure lower RMSE) = {probability_improved:.3f}',
        transform=axis.transAxes, ha='right', va='top',
        bbox={'facecolor': 'white', 'alpha': 0.85, 'edgecolor': 'none'},
    )
    axis.grid(axis='y', alpha=0.25)
    axis.legend(frameon=False)
axes[0].set_ylabel('Bootstrap count')
figure.suptitle(
    'Paired bootstrap distributions of crop-pure RMSE improvement',
    fontsize=14, fontweight='bold',
)
figure.tight_layout()
figure.savefig(
    OUTPUT_DIR / 'bootstrap_rmse_improvement_distribution.pdf',
    bbox_inches='tight',
)
plt.show()

## Save Models and Reproducibility Metadata

In [ ]:
for key in selected_model_keys:
    joblib.dump(
        fitted_models[key],
        OUTPUT_DIR / f'yield_model_{key}.joblib',
    )

metadata = {
    'research_question': (
        'To what extent does high-resolution, crop-pure enhanced SIF '
        'improve winter-wheat yield prediction RMSE compared with '
        'mixed-pixel SIF derived from raw satellite observations?'
    ),
    'input_file': str(DATA_PATH),
    'output_directory': str(OUTPUT_DIR),
    'target': 'ww_yield',
    'target_unit': 'metric tonnes/ha',
    'train_years': TRAIN_YEARS,
    'test_year': TEST_YEAR,
    'n_train': len(train_data),
    'n_test': len(test_data),
    'model': 'RandomForestRegressor',
    'fixed_model_settings': fixed_model_settings,
    'shared_control_feature_columns': shared_feature_columns,
    'selected_model_keys': selected_model_keys,
    'selected_model_features': {
        key: model_specs[key]['features'] for key in selected_model_keys
    },
    'bootstrap_repetitions': BOOTSTRAP_REPETITIONS,
    'random_seed': SEED,
}
with (OUTPUT_DIR / 'model_metadata.json').open('w', encoding='utf-8') as file:
    json.dump(metadata, file, indent=2)

print('Saved outputs:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(' -', path)